In [1]:
!pip install transformers accelerate pandas scikit-learn -q

In [2]:
from google.colab import files
uploaded = files.upload()

Saving imdb_test.csv to imdb_test.csv
Saving imdb_train.csv to imdb_train.csv
Saving imdb_val.csv to imdb_val.csv


In [3]:
import os
print(os.listdir())

['.config', 'imdb_test.csv', 'drive', 'imdb_train.csv', 'imdb_val.csv', 'sample_data']


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os
print(os.listdir())

['.config', 'imdb_test.csv', 'drive', 'imdb_train.csv', 'imdb_val.csv', 'sample_data']


In [6]:
import json
import time
from pathlib import Path
from statistics import mean, median

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

def compute_classification_metrics(y_true, y_pred) -> dict:
    cm = confusion_matrix(y_true, y_pred).tolist()
    return {
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "precision": round(float(precision_score(y_true, y_pred)), 4),
        "recall": round(float(recall_score(y_true, y_pred)), 4),
        "f1": round(float(f1_score(y_true, y_pred)), 4),
        "confusion_matrix": cm,
    }

def measure_inference_latency(predict_fn, samples, n_repeats=100, warmup=10) -> dict:
    for i in range(warmup):
        predict_fn(samples[i % len(samples)])
    timings_ms = []
    for i in range(n_repeats):
        text = samples[i % len(samples)]
        start = time.perf_counter()
        predict_fn(text)
        timings_ms.append((time.perf_counter() - start) * 1000)
    timings_sorted = sorted(timings_ms)
    p95_idx = int(len(timings_sorted) * 0.95)
    return {
        "n_measured": n_repeats,
        "mean_ms": round(mean(timings_ms), 3),
        "median_ms": round(median(timings_ms), 3),
        "p95_ms": round(timings_sorted[min(p95_idx, len(timings_sorted) - 1)], 3),
        "min_ms": round(min(timings_ms), 3),
        "max_ms": round(max(timings_ms), 3),
    }

def get_model_size_mb(path) -> float:
    path = Path(path)
    if path.is_dir():
        total_bytes = sum(f.stat().st_size for f in path.rglob("*") if f.is_file())
    else:
        total_bytes = path.stat().st_size
    return round(total_bytes / (1024 * 1024), 3)

BASE_MODEL = "distilbert-base-uncased"
RANDOM_STATE = 42
MAX_LENGTH = 512
NUM_EPOCHS = 3
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01
N_LATENCY_SAMPLES = 100

# --- Saved directly to Drive so a disconnect can't wipe results ---
DRIVE_DIR = Path("/content/drive/MyDrive/sentrisense")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUT_DIR = DRIVE_DIR / "distilbert_v1"
REPORT_OUT = DRIVE_DIR / "model3_distilbert_metrics.json"

set_seed(RANDOM_STATE)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[device] {device}" + (f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else " -- WARNING: no GPU, check Runtime settings"))

batch_size = 16 if device == "cuda" else 4
use_fp16 = device == "cuda"
print(f"[config] batch_size={batch_size}  fp16={use_fp16}  max_length={MAX_LENGTH}  epochs={NUM_EPOCHS}")

print("[load] reading uploaded CSVs...")
train_df = pd.read_csv("imdb_train.csv")
val_df = pd.read_csv("imdb_val.csv")
test_df = pd.read_csv("imdb_test.csv")
print(f"[load] train={len(train_df)} val={len(val_df)} test={len(test_df)}")

X_train_text = train_df["text_clean_transformer"].fillna("")
y_train = train_df["label"].tolist()
X_val_text = val_df["text_clean_transformer"].fillna("")
y_val = val_df["label"].tolist()
X_test_text = test_df["text_clean_transformer"].fillna("")
y_test = test_df["label"].tolist()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

class SentimentDataset:
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(list(texts), truncation=True, max_length=MAX_LENGTH)
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

print("[tokenize] encoding train/val...")
train_dataset = SentimentDataset(X_train_text, y_train, tokenizer)
val_dataset = SentimentDataset(X_val_text, y_val, tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)
param_count = sum(p.numel() for p in model.parameters())
print(f"[model] parameter count: {param_count:,}")

def compute_metrics_for_trainer(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return compute_classification_metrics(labels, preds)

training_args = TrainingArguments(
    output_dir="checkpoints",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=use_fp16,
    seed=RANDOM_STATE,
    logging_steps=100,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics_for_trainer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

print(f"\n[train] fine-tuning for up to {NUM_EPOCHS} epochs on {device}...")
t0 = time.perf_counter()
trainer.train()
training_time_s = time.perf_counter() - t0
print(f"[train] done in {training_time_s:.1f}s")

per_epoch_val_metrics = [
    {k: v for k, v in log.items() if k.startswith("eval_")}
    for log in trainer.state.log_history
    if "eval_f1" in log
]
for i, m in enumerate(per_epoch_val_metrics, start=1):
    print(f"  epoch {i}: val_acc={m.get('eval_accuracy')}  val_f1={m.get('eval_f1')}")

best_val_metrics = trainer.evaluate(val_dataset)
print(f"[best checkpoint] val_acc={best_val_metrics.get('eval_accuracy')}  val_f1={best_val_metrics.get('eval_f1')}")

print("\n[test] evaluating best checkpoint on held-out test set (ONE time)...")
test_dataset = SentimentDataset(X_test_text, y_test, tokenizer)
test_eval_raw = trainer.evaluate(test_dataset)
test_metrics = {
    "accuracy": round(float(test_eval_raw.get("eval_accuracy", 0.0)), 4),
    "precision": round(float(test_eval_raw.get("eval_precision", 0.0)), 4),
    "recall": round(float(test_eval_raw.get("eval_recall", 0.0)), 4),
    "f1": round(float(test_eval_raw.get("eval_f1", 0.0)), 4),
    "confusion_matrix": test_eval_raw.get("eval_confusion_matrix"),
}
print(f"[test] acc={test_metrics['accuracy']}  f1={test_metrics['f1']}")

print("\n[serialize] saving best checkpoint to Google Drive...")
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(MODEL_OUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUT_DIR))
disk_size_mb = get_model_size_mb(MODEL_OUT_DIR)
print(f"[serialize] saved -> {MODEL_OUT_DIR} ({disk_size_mb} MB)")

print("\n[measure] CPU inference latency (mandatory — this is the deployment-relevant number)...")
cpu_model = model.to("cpu")
cpu_model.eval()
cpu_samples = X_val_text.tolist()

def predict_single_cpu(text: str) -> int:
    inputs = tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    with torch.no_grad():
        logits = cpu_model(**inputs).logits
    return int(torch.argmax(logits, dim=-1).item())

cpu_latency = measure_inference_latency(predict_single_cpu, cpu_samples, n_repeats=N_LATENCY_SAMPLES)
print(f"[measure] CPU mean={cpu_latency['mean_ms']}ms  p95={cpu_latency['p95_ms']}ms")

gpu_latency = None
if device == "cuda":
    print("[measure] GPU inference latency (secondary figure only)...")
    gpu_model = model.to("cuda")
    gpu_model.eval()
    def predict_single_gpu(text: str) -> int:
        inputs = tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to("cuda")
        with torch.no_grad():
            logits = gpu_model(**inputs).logits
        return int(torch.argmax(logits, dim=-1).item())
    gpu_latency = measure_inference_latency(predict_single_gpu, cpu_samples, n_repeats=N_LATENCY_SAMPLES)
    print(f"[measure] GPU mean={gpu_latency['mean_ms']}ms  p95={gpu_latency['p95_ms']}ms")
    model.to("cpu")

report = {
    "model_name": "distilbert_finetuned",
    "model_version": "v1.0",
    "base_model": BASE_MODEL,
    "protocol_note": (
        "Trained on Google Colab (T4 GPU) due to no local GPU being available. "
        "Same fixed seed, same data split, same protocol as the local project scripts: "
        "validation drove epoch-level checkpoint selection and early stopping; test was "
        "evaluated exactly once, at the end, on the best checkpoint."
    ),
    "training_config": {
        "num_epochs_max": NUM_EPOCHS,
        "epochs_actually_run": len(per_epoch_val_metrics),
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "batch_size": batch_size,
        "max_length": MAX_LENGTH,
        "mixed_precision_fp16": use_fp16,
        "early_stopping_patience": 1,
        "device_used_for_training": device,
        "random_seed": RANDOM_STATE,
    },
    "per_epoch_validation_metrics": per_epoch_val_metrics,
    "best_checkpoint_validation_metrics": {
        "accuracy": round(float(best_val_metrics.get("eval_accuracy", 0.0)), 4),
        "precision": round(float(best_val_metrics.get("eval_precision", 0.0)), 4),
        "recall": round(float(best_val_metrics.get("eval_recall", 0.0)), 4),
        "f1": round(float(best_val_metrics.get("eval_f1", 0.0)), 4),
        "confusion_matrix": best_val_metrics.get("eval_confusion_matrix"),
    },
    "final_test_metrics": test_metrics,
    "training_time_s": round(training_time_s, 1),
    "parameter_count": param_count,
    "disk_size_mb": disk_size_mb,
    "inference_latency_cpu_val_samples": cpu_latency,
    "inference_latency_gpu_val_samples_SECONDARY_ONLY": gpu_latency,
    "note": "CPU latency is the deployment-relevant figure (AWS Lambda has no GPU). "
            "GPU latency is informational only.",
}

REPORT_OUT.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"\n[saved] Full report -> {REPORT_OUT}")
print(f"[saved] Model files -> {MODEL_OUT_DIR}")

[device] cuda (Tesla T4)
[config] batch_size=16  fp16=True  max_length=512  epochs=3
[load] reading uploaded CSVs...
[load] train=22389 val=2488 test=25000


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

[best checkpoint] val_acc=0.9188  val_f1=0.919

[test] evaluating best checkpoint on held-out test set (ONE time)...


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Confusion Matrix
0.158810,0.198851,2,0.931200,0.931200,0.931300,0.931200,"[[11640, 860], [859, 11641]]"


[test] acc=0.9312  f1=0.9312

[serialize] saving best checkpoint to Google Drive...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[serialize] saved -> /content/drive/MyDrive/sentrisense/distilbert_v1 (256.109 MB)

[measure] CPU inference latency (mandatory — this is the deployment-relevant number)...
[measure] CPU mean=307.072ms  p95=658.958ms
[measure] GPU inference latency (secondary figure only)...
[measure] GPU mean=8.363ms  p95=10.398ms

[saved] Full report -> /content/drive/MyDrive/sentrisense/model3_distilbert_metrics.json
[saved] Model files -> /content/drive/MyDrive/sentrisense/distilbert_v1
